# Spiking Neural Network (SNN) Violence Detection & Clip Assembly
## Google Colab Training & Evaluation Notebook

This thin wrapper notebook mounts Google Drive, installs dependencies (`spikingjelly`), and launches training and evaluation scripts.

In [ ]:
# Step 1: Mount Google Drive and Set Up Repository
import os
from google.colab import drive
drive.mount('/content/drive')

# Change directory to repo location on Drive
repo_dir = '/content/drive/MyDrive/snn-evidence'
if os.path.exists(repo_dir):
    %cd {repo_dir}
else:
    print(f'Repo directory not found at {repo_dir}. Please clone or copy repo to Drive.')

In [ ]:
# Step 2: Run Setup Script (Installs SpikingJelly and verifies GPU)
!python configs/colab_setup.py

In [ ]:
# Step 3: Run Smoke Tests
!python test_smoke.py

In [ ]:
# Step 4: Download Datasets (RWF-2000, Hockey Fight, RLVS)
# Note: Place your Kaggle API key (kaggle.json) in ~/.kaggle/ first
!python data/download_rwf.py --datasets rwf2000 hockey rlvs

In [ ]:
# Step 5: Preprocess Datasets (Resample to 16 fps, 112x112, frame-difference)
!python data/prep_rwf.py --dataset rwf2000 --encoding diff

In [ ]:
# Step 6: Train SNN Model on RWF-2000
!python train.py --model snn --epochs 50 --batch-size 16 --lr 0.001 --resume

In [ ]:
# Step 7: Train ANN Baseline Models (MobileNet+GRU, MobileNet+Pool, ANNMirrorNet)
!python train.py --model MobileNetGRU --epochs 30 --batch-size 16
!python train.py --model MobileNetPool --epochs 30 --batch-size 16
!python train.py --model ANNMirrorNet --epochs 30 --batch-size 16

In [ ]:
# Step 8: Evaluate Classification Accuracy & Cross-Dataset Generalization
!python eval_cls.py --model snn --checkpoint checkpoints/snn_best.pth --dataset rwf2000
!python eval_cls.py --model snn --checkpoint checkpoints/snn_best.pth --dataset hockey

In [ ]:
# Step 9: Evaluate Incident Boundary Detection & Hysteresis Ablation
!python data/make_streams.py --config configs/default.yaml
!python eval_boundary.py --checkpoint checkpoints/snn_best.pth --streams-dir data/streams --detector incident_neuron

In [ ]:
# Step 10: Run SOP Energy Estimation & Plot Comparison
!python energy.py --checkpoint checkpoints/snn_best.pth